# **Google Drive и библиотеки**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install faster-whisper jiwer librosa soundfile tqdm -q
!pip install noisereduce
!pip install pyloudnorm

import os
import tarfile
import time
import json
import numpy as np
from faster_whisper import WhisperModel
import librosa
import jiwer
import torch
from tqdm import tqdm

print("✅ Библиотеки установлены, Google Drive смонтирован")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.5 MB/s eta 0:00:00
✅ Библиотеки установлены, Google Drive смонтирован


# **Параметры пайплайна**

In [ ]:
QUANTIZATION = "int8" #default int8
MODEL_SIZE = "large-v3" #large-v3 large-v2
VAD_ENABLED = True # Enables Silero VAD
DENOISING_ENABLED = True # Enables Audio Denoising
LOUDNESS_NORMALIZATION_EBABLED = True # Enables Volume Normaliztion

MAX_AUDIO_FILES = 100 # Audio Files Count to Evaluate Metrics

# **Загружаем модель faster-whisper**

In [ ]:
print("🚀 Загрузка модели faster-whisper...")

# Определяем устройство
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"📊 Модель: {MODEL_SIZE}")
print(f"💻 Устройство: {device}")

# Загружаем модель
model = WhisperModel(
    MODEL_SIZE,
    device=device,
    compute_type=QUANTIZATION,
    download_root="/content/models"
)

print("✅ Модель загружена")

🚀 Загрузка модели faster-whisper...
📊 Модель: large-v3
💻 Устройство: cuda
✅ Модель загружена


# **Очистка текста**

In [ ]:
import re

def clean_text(text):
    # Удаляем все знаки препинания, оставляем только буквы, цифры и пробелы
    # Для русского языка важно сохранить букву 'ё'
    text = re.sub(r'[^\w\s]', '', text)  # Удаляет всё, кроме букв, цифр и пробелов
    text = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)  # Unicode-версия
    return text.lower().strip()

# **Распаковка одного аудио файла**

In [ ]:
# Укажите путь к вашему архиву
archive_path = "/content/drive/MyDrive/ruls_data.tar.gz"

# Создаем временную папку
os.makedirs("/content/sample_audio", exist_ok=True)

# Открываем архив и ищем первый аудио файл из test-сета
with tarfile.open(archive_path, 'r:gz') as tar:
    # Получаем список всех файлов
    members = tar.getmembers()

    # Ищем первый аудио файл из папки test
    audio_file = None
    text = ""

    for member in members:
        # Ищем файлы в test/audio/
        if member.name.startswith('test/audio/') and member.name.endswith('.wav'):
            audio_file = member
            print(f"🎵 Найден тестовый аудио файл: {member.name}")
            break

    tar.extract(audio_file, path="/content/sample_audio")
    audio_path = f"/content/sample_audio/{audio_file.name}"

    # Ищем соответствующий текст в manifest.json
    # Путь к manifest.json для test-сета
    manifest_path = None
    for member in members:
        if member.name == 'test/manifest.json':
            manifest_path = member
            break

    if manifest_path:
        # Извлекаем manifest.json
        tar.extract(manifest_path, path="/content/sample_audio")

        # Читаем manifest
        with open('/content/sample_audio/test/manifest.json', 'r', encoding='utf-8') as f:
            for line in f:
                manifest_entry = json.loads(line)
                # Сравниваем имя файла
                if audio_file.name.split('/')[-1] in manifest_entry['audio_filepath']:
                    preprocessed_text = manifest_entry['text']
                    original_text = manifest_entry['text_no_preprocessing']
                    print(f"📝 Найден текст в manifest:")
                    print(f"   Предобработанный: {preprocessed_text[:100]}..." if len(text) > 100 else f"   Предобработанный: {preprocessed_text}")
                    print(f"   Оригинальный: {original_text[:100]}..." if len(original_text) > 100 else f"   Оригинальный: {original_text}")
                    break

🎵 Найден тестовый аудио файл: test/audio/4372/2145/poemi_15_pushkin_0093.wav


/tmp/ipython-input-3989110990.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(audio_file, path="/content/sample_audio")
/tmp/ipython-input-3989110990.py:36: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(manifest_path, path="/content/sample_audio")


📝 Найден текст в manifest:
   Предобработанный: уходит и поет старый муж и проч
   Оригинальный: Уходит и поет: Старый муж и проч.


# **Считывание аудио**

In [ ]:
audio, sr = librosa.load(audio_path, sr=None)
audio_duration = len(audio) / sr
print(f"⏱️ Длительность аудио: {audio_duration:.2f} сек")

⏱️ Длительность аудио: 3.12 сек


# **Noise Reduction**

In [ ]:
import noisereduce as nr

def denoise_audio(audio, sr, stationary=True, prop_decrease=0.9):
    """
    Simple universal noise reduction function.

    Parameters:
    -----------
    audio : numpy.ndarray
        Audio time series as numpy array (from librosa.load)
    sr : int
        Sampling rate of the audio
    stationary : bool, optional (default=True)
        If True, assumes noise is stationary (constant background noise).
        If False, adapts to changing noise characteristics
    prop_decrease : float, optional (default=0.9)
        How much noise to remove (0.0 to 1.0).
        1.0 removes all estimated noise, 0.0 removes nothing

    Returns:
    --------
    numpy.ndarray
        Cleaned audio time series with the same shape as input
    """

    # Apply noise reduction
    # The algorithm automatically estimates noise profile from the audio spectrum
    cleaned_audio = nr.reduce_noise(
        y=audio,
        sr=sr,
        stationary=stationary,
        prop_decrease=prop_decrease
    )

    return cleaned_audio

# **Loudness Normalization**

In [ ]:
import numpy as np
import pyloudnorm as pyln

def normalize_loudness(
    audio: np.ndarray,
    sr: int,
    target_loudness: float = -23.0,
    method: str = 'lufs'
) -> np.ndarray:
    """
    Normalize audio loudness to a target level.

    This function applies loudness normalization to ensure consistent
    volume levels across different audio files, which improves ASR performance.

    Parameters
    ----------
    audio : np.ndarray
        Audio time series as a numpy array (values typically in range [-1, 1])
    sr : int
        Sampling rate of the audio in Hz
    target_loudness : float, optional (default=-23.0)
        Target loudness level in LUFS (Loudness Units relative to Full Scale)
        Common values:
        - -23.0 LUFS: Broadcast standard (EBU R128)
        - -16.0 LUFS: Music streaming standard
        - -14.0 LUFS: Podcast standard
    method : str, optional (default='lufs')
        Normalization method:
        - 'lufs': ITU-R BS.1770 loudness normalization (recommended)
        - 'peak': Simple peak normalization
        - 'rms': Root mean square normalization

    Returns
    -------
    np.ndarray
        Loudness-normalized audio array with the same shape as input

    Examples
    --------
    >>> import librosa
    >>> audio, sr = librosa.load('speech.wav', sr=16000)
    >>> audio_norm = normalize_loudness(audio, sr, target_loudness=-23.0)
    >>> # Save normalized audio
    >>> import soundfile as sf
    >>> sf.write('speech_norm.wav', audio_norm, sr)

    Notes
    -----
    - LUFS normalization is perceptually optimal and recommended for ASR
    - The function prevents clipping by applying a limiter when necessary
    - Very quiet files fall back to peak normalization automatically
    """

    # Handle empty or silent audio
    if len(audio) == 0 or np.max(np.abs(audio)) == 0:
        return audio

    if method == 'lufs':
        meter = pyln.Meter(sr)
        current_loudness = meter.integrated_loudness(audio)

        # Check for invalid loudness measurement (silent or near-silent audio)
        if np.isnan(current_loudness) or current_loudness == -np.inf:
            print("Info: Audio too quiet, falling back to peak normalization")
            return _peak_normalize(audio)

        # Apply loudness normalization
        audio_normalized = pyln.normalize.loudness(
            audio,
            current_loudness,
            target_loudness
        )

    elif method == 'peak':
        audio_normalized = _peak_normalize(audio)

    elif method == 'rms':
        audio_normalized = _rms_normalize(audio, target_loudness)

    else:
        raise ValueError(f"Unknown method: {method}. Use 'lufs', 'peak', or 'rms'")

    # Ensure no clipping by limiting to [-1, 1] range
    audio_normalized = np.clip(audio_normalized, -1.0, 1.0)

    return audio_normalized.astype(np.float32)


def _peak_normalize(audio: np.ndarray, target_peak: float = 0.95) -> np.ndarray:
    """
    Internal function for peak normalization.

    Parameters
    ----------
    audio : np.ndarray
        Input audio array
    target_peak : float, optional (default=0.95)
        Target peak amplitude (should be < 1.0 to avoid clipping)

    Returns
    -------
    np.ndarray
        Peak-normalized audio
    """
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        return audio * (target_peak / max_val)
    return audio


def _rms_normalize(audio: np.ndarray, target_rms: float = 0.1) -> np.ndarray:
    """
    Internal function for RMS normalization.

    Parameters
    ----------
    audio : np.ndarray
        Input audio array
    target_rms : float, optional (default=0.1)
        Target RMS value (typical range: 0.05-0.2)

    Returns
    -------
    np.ndarray
        RMS-normalized audio
    """
    current_rms = np.sqrt(np.mean(audio**2))
    if current_rms > 0:
        return audio * (target_rms / current_rms)
    return audio


def get_loudness(audio: np.ndarray, sr: int) -> float:
    """
    Measure the integrated loudness of an audio signal in LUFS.

    Parameters
    ----------
    audio : np.ndarray
        Audio time series
    sr : int
        Sampling rate

    Returns
    -------
    float
        Integrated loudness in LUFS, or -np.inf for silent audio
    """
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio)
    return loudness


def is_audio_too_quiet(audio: np.ndarray, sr: int, threshold: float = -40.0) -> bool:
    """
    Check if audio is too quiet for reliable ASR.

    Parameters
    ----------
    audio : np.ndarray
        Audio time series
    sr : int
        Sampling rate
    threshold : float, optional (default=-40.0)
        Loudness threshold in LUFS (audio below this is considered too quiet)

    Returns
    -------
    bool
        True if audio is too quiet, False otherwise
    """
    loudness = get_loudness(audio, sr)
    return loudness < threshold

# **Транскрибирование тестового файла**

In [ ]:
print("🎤 Транскрибируем тестовый аудио файл...")

if DENOISING_ENABLED:
    audio = denoise_audio(audio, sr)

if LOUDNESS_NORMALIZATION_EBABLED:
    audio = normalize_loudness(audio, sr)

# Время старта
start_time = time.time()

# Транскрибация
segments, _ = model.transcribe(
    audio=audio,
    language="ru",
    vad_filter=VAD_ENABLED,
    beam_size=5
)

# Собираем текст
hypothesis_text = " ".join([segment.text for segment in segments])
hypothesis_text = clean_text(hypothesis_text)

# Время транскрибации
trans_time = time.time() - start_time
rtf = trans_time / audio_duration

print(f"⏱️ Время обработки: {trans_time:.2f} сек")
print(f"📊 RTF: {rtf:.3f}")

print("\n📄 ПОЛУЧЕННАЯ ТРАНСКРИПЦИЯ:")
print("-" * 50)
print(hypothesis_text)
print("-" * 50)

# Считаем WER
if preprocessed_text:
    wer = jiwer.wer(preprocessed_text, hypothesis_text)
    print(f"\n📊 WER (Word Error Rate): {wer:.2%}")
else:
    print("\n⚠️ Нет reference текста для расчета WER")

🎤 Транскрибируем тестовый аудио файл...
⏱️ Время обработки: 0.82 сек
📊 RTF: 0.263

📄 ПОЛУЧЕННАЯ ТРАНСКРИПЦИЯ:
--------------------------------------------------
уходит и поет старый муж и прочь
--------------------------------------------------

📊 WER (Word Error Rate): 14.29%


# **Метрики на всем датасете**

In [ ]:
print("📊 ЗАПУСК ОЦЕНКИ НА TEST-СЕТЕ")
print("=" * 60)

# Счетчики для метрик
all_wer = []
all_rtf = []
all_audio_durations = []
file_count = 0

# Открываем архив
with tarfile.open(archive_path, 'r:gz') as tar:
    # Получаем все файлы
    members = tar.getmembers()

    # Ищем manifest.json для test-сета
    manifest_member = None
    for member in members:
        if member.name == 'test/manifest.json':
            manifest_member = member
            break

    # Извлекаем manifest
    tar.extract(manifest_member, path="/content/temp")

    # Читаем все записи из manifest
    test_files = []
    with open('/content/temp/test/manifest.json', 'r', encoding='utf-8') as f:
        for line in f:
            test_files.append(json.loads(line))

    print(f"📊 Найдено {len(test_files)} тестовых файлов в manifest")

    # Ограничим количество для скорости (можно убрать ограничение)
    max_files = min(MAX_AUDIO_FILES, len(test_files))  # ⬅️ Увеличьте для полной оценки
    print(f"🔍 Будет обработано {max_files} файлов")

    # Обрабатываем каждый файл из manifest
    for i, file_info in enumerate(tqdm(test_files[:max_files], desc="Обработка файлов")):
        try:
            # Получаем путь к аудио файлу из manifest
            audio_relative_path = file_info['audio_filepath']
            audio_filename = os.path.basename(audio_relative_path)

            # Ищем соответствующий аудио файл в архиве
            audio_member = None
            for member in members:
                if member.name.endswith(audio_filename):
                    audio_member = member
                    break

            if not audio_member:
                continue

            # Извлекаем аудио
            tar.extract(audio_member, path="/content/temp_audio")
            audio_path = f"/content/temp_audio/{audio_member.name}"

            # print(f"Путь к аудио файлу: {audio_path}")

            audio_duration = len(audio) / sr
            # print(f"⏱️ Длительность аудио: {audio_duration:.2f} сек")
            all_audio_durations.append(audio_duration)

            # Текст из manifest
            preprocessed_text = file_info['text']
            original_text = file_info['text_no_preprocessing']

            # print(f"📝 Original Text Reference: {original_text}")
            # print(f"📝 Preprocessed Text Reference: {preprocessed_text}")

            # Читаем аудио
            audio, sr = librosa.load(audio_path, sr=None)

            if DENOISING_ENABLED:
                audio = denoise_audio(audio, sr)

            if LOUDNESS_NORMALIZATION_EBABLED:
                audio = normalize_loudness(audio, sr)

            # Транскрибация
            start_time = time.time()

            segments, _ = model.transcribe(
                audio=audio,
                language="ru",
                vad_filter=VAD_ENABLED,
                beam_size=5
            )

            hypothesis = " ".join([seg.text for seg in segments])
            hypothesis = clean_text(hypothesis)
            trans_time = time.time() - start_time

            # Длительность из manifest
            duration = file_info['duration']

            # Метрики
            wer = jiwer.wer(preprocessed_text, hypothesis)
            rtf = trans_time / duration

            all_wer.append(wer)
            all_rtf.append(rtf)
            file_count += 1

            # Очищаем временные файлы
            os.remove(audio_path)

        except Exception as e:
            print(f"\n❌ Ошибка при обработке файла {i}: {e}")
            continue

    # Очищаем временные папки
    os.system("rm -rf /content/temp_audio")
    os.system("rm -rf /content/temp")


print("\n" + "=" * 60)
print("📊 ИТОГОВЫЕ МЕТРИКИ НА TEST-СЕТЕ")
print("=" * 60)
print(f"Обработано файлов: {file_count}")

if all_wer:
    avg_wer = np.mean(all_wer)
    avg_rtf = np.mean(all_rtf)
    avg_audio_duration = np.mean(all_audio_durations)

    print(f"\n📈 Средняя длина аудиозаписи: {avg_audio_duration:.2f} сек")


    print(f"\n🎯 WER (Word Error Rate): {avg_wer:.2%}")
    print(f"⚡ RTF (Real Time Factor): {avg_rtf:.3f}")
else:
    print("❌ Нет данных для расчета метрик")

📊 ЗАПУСК ОЦЕНКИ НА TEST-СЕТЕ


/tmp/ipython-input-3621564265.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(manifest_member, path="/content/temp")


📊 Найдено 1352 тестовых файлов в manifest
🔍 Будет обработано 100 файлов


Обработка файлов:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipython-input-3621564265.py:55: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(audio_member, path="/content/temp_audio")


⏱️ Длительность аудио: 3.12 сек
📝 Original Text Reference: Для вас, души моей царицы, Красавицы, для вас одних Времен минувших небылицы, В часы досугов золотых, Под шепот старины болтливой, Рукою верной я писал
📝 Preprocessed Text Reference: для вас души моей царицы красавицы для вас одних времен минувших небылицы в часы досугов золотых под шепот старины болтливой рукою верной я писал


...

⏱️ Длительность аудио: 3.28 сек
📝 Original Text Reference: Ревнивый, трепетный хранитель Замков безжалостных дверей, Он только немощный мучитель Прелестной пленницы своей.
📝 Preprocessed Text Reference: ревнивый трепетный хранитель замков безжалостных дверей он только немощный мучитель прелестной пленницы своей


Обработка файлов:  99%|█████████▉| 99/100 [04:42<00:03,  3.38s/it]

⏱️ Длительность аудио: 7.85 сек
📝 Original Text Reference: Вокруг нее он молча бродит, Клянет жестокий жребий свой.
📝 Preprocessed Text Reference: вокруг нее он молча бродит клянет жестокий жребий свой


Обработка файлов: 100%|██████████| 100/100 [04:46<00:00,  2.87s/it]


📊 ИТОГОВЫЕ МЕТРИКИ НА TEST-СЕТЕ
Обработано файлов: 100

📈 Средняя длина аудиозаписи: 5.94 сек

🎯 WER (Word Error Rate): 29.37%
⚡ RTF (Real Time Factor): 0.188


In [ ]:
# @title Очистка временных файлов
print("🧹 Очистка временных файлов...")
os.system("rm -rf /content/sample_audio")
os.system("rm -rf /content/temp_audio")
os.system("rm -rf /content/temp")
os.system("rm -rf /content/models")
print("✅ Готово!")

🧹 Очистка временных файлов...
✅ Готово!
